<a href="https://colab.research.google.com/github/Sagaustus/adh-group-projects/blob/main/group-01-african-films/analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A Title, a Country, an Identifier

### Thin records and the labour geography of African cinema on Wikidata

**Group 1 · working chapter draft**

---

This notebook runs the analysis end to end. It produces the figure and the numbers
your chapter needs. What it does **not** do is decide what they mean — the cells
marked **YOUR DECISION** are where your judgement enters, and they are the only
part a reader will credit to you.

**The thesis you are testing.** Wikidata's record of African cinema is not thin
because African cinema is thin. It is thin in a *patterned* way, and the pattern
follows the labour and sources of the editors rather than the films.

In [ ]:
# Setup — run this first. Nothing to upload.
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

URL = "https://raw.githubusercontent.com/Sagaustus/adh-dh-datasets/main/datasets/09_african_films/data.csv"
df = pd.read_csv(URL)
print(f"{len(df):,} rows x {df.shape[1]} columns")
print(df.dtypes.to_string())

## Step 1 · Frame

The assignment asks for one descriptive and one analytical question. Here they are,
with the method each needs.

| | Question | Method |
|---|---|---|
| **Descriptive** | How complete is the record of African cinema on Wikidata, and does completeness vary by country? | Proportions and a grouped comparison |
| **Analytical** | Is country a predictor of record completeness once we control for how many films a country has and when they were made? | Logistic regression |

The second question exists because the first has an obvious trap, and the next cell
walks straight into it so you can see it.

In [ ]:
# Define the outcome: a record is "complete" if it names both a genre and a director.
df["complete"] = df["genre"].notna() & df["director"].notna()

print(f"overall completeness: {df['complete'].mean():.1%}")
print(f"records with NEITHER genre nor director: {(~df['genre'].notna() & ~df['director'].notna()).sum():,}")
print()

# The trap: rank countries by completeness without a minimum count.
naive = df.groupby("country")["complete"].agg(["size", "mean"]).sort_values("mean", ascending=False)
print("Top 8 countries by completeness — WITHOUT a minimum count:")
print(naive.head(8).to_string(float_format=lambda v: f"{v:.2f}"))

Read that table before going on. The countries at the top have perfect completeness
**on a handful of records each**. A country with two films, both complete, scores
1.00 and tells you nothing.

This is the single most common error in dataset work: a rate computed over a
denominator too small to mean anything. Fix it by setting a floor — and by stating
the floor in your chapter, because it is a decision you made.

In [ ]:
MIN_FILMS = 30          # YOUR DECISION — and report it in the chapter

counts = df["country"].value_counts()
keep = counts[counts >= MIN_FILMS].index
sub = df[df["country"].isin(keep)].copy()

print(f"{len(keep)} countries have at least {MIN_FILMS} films "
      f"({len(sub):,} of {len(df):,} records retained)")
print()
by_country = (sub.groupby("country")["complete"]
                .agg(films="size", completeness="mean")
                .sort_values("completeness", ascending=False))
print(by_country.to_string(float_format=lambda v: f"{v:.3f}"))

## Step 2 · Absence audit

The assignment asks for a missing variable, a missing population, and a level of
detail that is too coarse. This dataset supplies all three without much searching.

In [ ]:
print("WHAT THIS DATASET NEVER RECORDED\n")
print("A missing VARIABLE")
print("   No gender field. The question 'how many of these films were directed by")
print("   women' cannot be asked of this table at all. Wikidata CAN express it")
print(f"   (property P21) for the {df['director'].nunique():,} named directors — so the")
print("   silence is in this export, not in the schema. That distinction is your")
print("   chapter's sharpest point: representable and unrepresented.\n")

print("A missing POPULATION")
top2 = df["country"].value_counts().head(2)
print(f"   {top2.index[0]} ({top2.iloc[0]:,}) and {top2.index[1]} ({top2.iloc[1]:,}) supply "
      f"{top2.sum()/len(df):.0%} of all records")
print(f"   across {df['country'].nunique()} countries. Lusophone and Sahelian cinema are")
print("   thin to the point of absence. Is that production reality, or sourcing?\n")

print("Detail too COARSE")
print("   `genre` is a free-text field with", f"{df['genre'].nunique():,} distinct values —")
print("   including near-duplicates. It is not a controlled vocabulary, so any")
print("   count by genre is a count of editors' phrasing.")
print()
print("   Example genre values:", df["genre"].dropna().unique()[:6].tolist())

**YOUR DECISION.** Which absence most limits your questions? Write two sentences.

The answer is not obvious and the choice shapes your chapter. The gender field is the
most rhetorically powerful; the country skew is the most measurable; the genre
vocabulary is the one that would sink a naive analysis fastest.

## Step 3 · Describe

The variable at the centre of the question is completeness. It is a proportion, so
the mean is the right summary — but the *distribution across countries* is what the
chapter is about, and that needs more than one number.

In [ ]:
print(f"completeness across the {len(by_country)} retained countries")
print(f"   mean   {by_country['completeness'].mean():.3f}")
print(f"   median {by_country['completeness'].median():.3f}")
print(f"   range  {by_country['completeness'].min():.3f} to {by_country['completeness'].max():.3f}")
print(f"   IQR    {by_country['completeness'].quantile(.25):.3f} to {by_country['completeness'].quantile(.75):.3f}")
print()
print("Mean and median are close here, so the country-level distribution is not badly")
print("skewed and either summary is defensible. Say which you used and why — the")
print("assignment asks for a deliberate choice, not a correct one.")
print()
# Completeness over time is the other axis worth describing.
by_decade = (df.dropna(subset=["year"])
               .assign(decade=lambda d: (d["year"] // 10 * 10).astype(int))
               .groupby("decade")["complete"].agg(films="size", completeness="mean"))
print(by_decade[by_decade["films"] >= 20].to_string(float_format=lambda v: f"{v:.3f}"))

## Step 4 · Compare

Split the data into two groups that matter. The obvious split is the two dominant
producers against everyone else — but state the difference in units a reader thinks
in, which here means percentage points, not odds.

In [ ]:
big = df["country"].value_counts().head(2).index.tolist()
a = df[df["country"].isin(big)]["complete"]
b = df[~df["country"].isin(big) & df["country"].isin(keep)]["complete"]

print(f"{' + '.join(big)}: {a.mean():.1%} complete  (n = {len(a):,})")
print(f"all other retained countries: {b.mean():.1%} complete  (n = {len(b):,})")
print(f"difference: {(a.mean() - b.mean())*100:+.1f} percentage points")
print()
print("Write that difference into your chapter in exactly those units. 'Records from")
print("the two largest producers are X percentage points less complete' is a sentence")
print("a reader can hold. An odds ratio is not.")

## Step 5 · Test

Could that difference be chance? The assignment requires a test **and** an effect
size. Then the analytical question needs the regression, because the raw comparison
confounds country with era and with catalogue size.

In [ ]:
from scipy.stats import chi2_contingency

table = pd.crosstab(df["country"].isin(big), df["complete"])
chi2, p, dof, expected = chi2_contingency(table)
n = table.values.sum()
phi = np.sqrt(chi2 / n)          # effect size for a 2x2 table

print(f"chi-square {chi2:.1f}, p = {p:.2e}")
print(f"phi (effect size) = {phi:.3f}")
print()
print("With 5,581 records almost any difference reaches significance — this is")
print("Kilgarriff's objection, and it is why phi is the number that matters here.")
print("Conventional reading: 0.1 small, 0.3 medium, 0.5 large.")

### First, a modelling trap worth failing in public

The obvious model is: completeness ~ country + year + catalogue size. It does not
work, and the reason is worth a paragraph in your chapter.

**Catalogue size is constant within a country.** Nigeria always has 1,668 films. So
once you include a dummy for every country, catalogue size is an exact linear
combination of those dummies — perfect collinearity, and the fit will not converge.
You cannot control for a country-level variable *and* include country fixed effects.

The fix is to decide which question you are asking, and run the matching model.

In [ ]:
import statsmodels.api as sm

model_df = sub.dropna(subset=["year"]).copy()
model_df["year_c"] = model_df["year"] - model_df["year"].mean()
y = model_df["complete"].astype(int).values

# MODEL A — "which countries differ, holding era constant?"
# Country dummies, NO catalogue size (it is collinear with them).
Xa = pd.get_dummies(model_df[["country"]], drop_first=True).astype(float)
Xa["year_c"] = model_df["year_c"].values
Xa = sm.add_constant(Xa)
fit_a = sm.Logit(y, Xa).fit(disp=0)
print(f"MODEL A  n = {len(y):,}   pseudo R2 = {fit_a.prsquared:.3f}   "
      f"converged = {fit_a.mle_retvals['converged']}")

# MODEL B — "does catalogue size predict completeness at all?"
# Catalogue size and era, NO country dummies.
model_df["log_size"] = np.log(model_df["country"].map(counts))
Xb = sm.add_constant(model_df[["log_size", "year_c"]].astype(float))
fit_b = sm.Logit(y, Xb).fit(disp=0)
print(f"MODEL B  n = {len(y):,}   pseudo R2 = {fit_b.prsquared:.3f}   "
      f"converged = {fit_b.mle_retvals['converged']}")
print()
print(pd.DataFrame({"coef": fit_b.params, "p": fit_b.pvalues,
                    "odds_ratio": np.exp(fit_b.params)}).to_string(
      float_format=lambda v: f"{v:.4f}"))

In [ ]:
# Model A's country effects, as evidence rather than as a score.
res = pd.DataFrame({"coef": fit_a.params, "p": fit_a.pvalues})
res["odds_ratio"] = np.exp(res["coef"])
res = res[res.index.str.startswith("country_")].copy()
res.index = res.index.str.replace("country_", "", regex=False)
res = res.sort_values("coef", ascending=False)

print("Country effects, relative to the omitted baseline country")
print("(positive = MORE complete records than baseline, holding era constant)\n")
print(pd.concat([res.head(6), res.tail(5)]).to_string(float_format=lambda v: f"{v:.3f}"))
print()
sig = (res["p"] < 0.05).sum()
print(f"{sig} of {len(res)} country effects reach p < 0.05.")
print("Report the coefficients, not just the count — module 28a's argument is that")
print("an interpretable model earns its keep by being quotable in a footnote.")

**YOUR DECISION.** Read the coefficients as evidence, not as a score.

- Does `year_c` matter? A positive coefficient means newer films are better
  documented — an argument about editor attention, not about cinema.
- Does `log_size` matter? If catalogue size predicts completeness, the effect is
  about editorial capacity rather than about any particular country.
- Which countries retain a large coefficient *after* those controls? Those are the
  cases your chapter has to explain, and they are where the qualitative work goes.

## Step 6 · Show

One chart. Sorted deliberately, with a reference line, and a caption saying what you
filtered. The chart below plots completeness against catalogue size, because that is
the relationship the regression is about — a bare ranking would hide it.

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6))
ax.scatter(by_country["films"], by_country["completeness"], s=70,
           color="#2b5c50", alpha=.85, zorder=3)
for name, r in by_country.iterrows():
    ax.annotate(name, (r["films"], r["completeness"]), fontsize=8,
                xytext=(5, 3), textcoords="offset points")

overall = df["complete"].mean()
ax.axhline(overall, ls="--", color="#9c2c1f", lw=1.2, zorder=2,
           label=f"overall completeness ({overall:.0%})")
ax.set_xscale("log")
ax.set_xlabel("films in the dataset (log scale)")
ax.set_ylabel("proportion of records with both genre and director")
ax.set_title("Record completeness against catalogue size, African cinema on Wikidata")
ax.legend(loc="lower right")
plt.tight_layout()

print(f"CAPTION. Countries with at least {MIN_FILMS} films (n = {len(by_country)} of "
      f"{df['country'].nunique()} countries, {len(sub):,} of {len(df):,} records).")
print("Completeness is the proportion of records naming both a genre and a director.")
print("Dashed line is the overall rate across the unfiltered dataset.")

## Step 7 · The qualitative half

The regression tells you **which** records are thin. It cannot tell you **why**, and
a chapter with only the quantitative half will be sent back.

The method is a small coding exercise, and it is the part that makes this publishable
rather than descriptive. Module 17's protocol applies: two coders, independently, and
report agreement.

In [ ]:
# Draw a purposive sample of thin records to code by hand.
thin = df[df["genre"].isna() & df["director"].isna()]
print(f"{len(thin):,} records have neither genre nor director "
      f"({len(thin)/len(df):.0%} of the dataset)\n")

sample = thin.sample(n=20, random_state=7)[["wikidata_id", "title", "country", "year"]]
print("Twenty to code by hand. Open each on Wikidata and read its edit history:\n")
for _, r in sample.iterrows():
    yr = "" if pd.isna(r["year"]) else int(r["year"])
    print(f"   https://www.wikidata.org/wiki/{r['wikidata_id']:<10}  "
          f"{str(r['title'])[:40]:<42}{r['country']:<16}{yr}")

### The coding scheme

Apply one label per record. These are a starting point — refine them after coding
ten, which is what a pilot is for.

| Label | Definition |
|---|---|
| `IMPORT` | Created by a bot or bulk import from an external catalogue; the thinness is the import's shape |
| `PLACEHOLDER` | Created by a human as a stub, with an evident intention to return |
| `ASSERTION` | Created by a human to record that the film *exists*, with no sources available to say more |
| `ORPHAN` | No clear provenance; edit history gives no account |

**Two coders, independently, then compute agreement.** A scheme two people cannot
apply consistently is not a finding, it is an opinion — and reviewers check this.

In [ ]:
from sklearn.metrics import cohen_kappa_score

# Replace with your real codes after both coders have worked through the twenty.
coder_a = ["IMPORT","IMPORT","ASSERTION","PLACEHOLDER","IMPORT","ORPHAN","IMPORT",
           "ASSERTION","IMPORT","IMPORT","PLACEHOLDER","ASSERTION","IMPORT","ORPHAN",
           "IMPORT","ASSERTION","IMPORT","IMPORT","PLACEHOLDER","IMPORT"]
coder_b = ["IMPORT","IMPORT","ASSERTION","IMPORT","IMPORT","ORPHAN","IMPORT",
           "PLACEHOLDER","IMPORT","IMPORT","PLACEHOLDER","ASSERTION","IMPORT","ASSERTION",
           "IMPORT","ASSERTION","IMPORT","ORPHAN","PLACEHOLDER","IMPORT"]

kappa = cohen_kappa_score(coder_a, coder_b)
raw = np.mean([x == y for x, y in zip(coder_a, coder_b)])
print(f"raw agreement {raw:.2f}   Cohen's kappa {kappa:.3f}")
print()
print("Landis & Koch: <0.20 slight, 0.21-0.40 fair, 0.41-0.60 moderate,")
print("0.61-0.80 substantial, >0.80 almost perfect.")
print()
print("Where you disagreed:")
for i, (x, y) in enumerate(zip(coder_a, coder_b)):
    if x != y:
        print(f"   record {i+1}: {x} vs {y}")
print()
print("A systematic disagreement is a problem with the SCHEME, not the coder.")
print("Revise the definition and re-code — that is a round, and rounds get reported.")

## Step 8 · Limits

The last page of the chapter, and the first thing a reviewer reads properly.

**What this analysis supports**

- Statements about *this export* of Wikidata, retrieved on its stated date.
- Statements about record completeness as defined here — genre and director both
  present. Another definition gives another number.
- A relationship between country, catalogue size and completeness, with the effect
  sizes reported above.

**What it does not support**

- Any claim about African film *production*. This is a catalogue, not a filmography.
- Any claim about women directors. The variable does not exist in this table.
- Any claim that a country's cinema is under-documented *because* of anything —
  the regression shows association, and the coding exercise offers a mechanism for
  thin records, not a cause for country differences.
- Any claim about Wikidata in general. African cinema is one corner of it.

**YOUR DECISION.** Add two more sentences this data does not support. Being able to
name them is what separates the chapter from a report — and reviewers who see a
short limits section assume you did not look.

## What to hand in

1. **One page**: the finding, the method, the uncertainty, and the list above.
2. **One chart**, captioned, saying what you filtered.
3. **The coding sheet** with both coders' labels and the kappa.

### Turning this into the chapter

The draft has a spine already: a measured pattern, a mechanism from the coding, and
a structural absence (gender) that is representable and unrepresented. What it still
needs from you is the **counter-practice** — someone, somewhere, is fixing this.
Language-community edit-a-thons, WikiProject Africa, AfroCROWD. A critique with no
counter-practice reads as a complaint; with one, it reads as an intervention.